# 03 — Feature engineering
Use the existing, reviewed topology implementation. The 22-column dataset represents each player's position at the end of setup. Pips are counts of equally likely dice outcomes (out of 36), not actual observed production. Expansion options assume a frozen initial board.

In [ ]:
from pathlib import Path
from importlib import import_module
import pandas as pd

OUT = Path("processed_data_5000")
fe = import_module("03_feature_engineering")
placements = pd.read_csv(OUT / "initial_placements.csv")
tiles = pd.read_csv(OUT / "map_tiles.csv")
features = fe.make_features(placements, tiles)
assert len(features) == len(placements)
assert not features.isna().any().any()
assert not features.duplicated(["Game_ID","Player"]).any()
assert features.Total_Production.eq(features.Settlement_1_Score + features.Settlement_2_Score).all()
production = [r.title()+"_Production" for r in fe.RESOURCES]
assert features.Total_Production.eq(features[production].sum(axis=1)).all()
assert features.Initial_Total.eq(features[["Initial_"+r.title() for r in fe.RESOURCES]].sum(axis=1)).all()
display(features.head())
print(features.shape)

In [ ]:
# Rebuild the feature CSV only if needed. This is a local file operation.
features.to_csv(OUT / "initial_features.csv", index=False)
display(features.select_dtypes("number").describe().T)

Excluded: final board, dice outcomes, winner, third settlement, player IDs as predictors. Port types are not part of the core feature CSV; they are a possible extension. Aggregate columns are retained for interpretation but redundant columns will be excluded from modeling.